# 習題：數學物件、Python 型別與 AI 中的數值表示

在數學中，我們經常接觸下列概念：

- 自然數（Natural Numbers）
- 整數（Integers）
- 有理數（Rational Numbers）
- 實數（Real Numbers）
- 複數（Complex Numbers）
- 向量（Vectors）
- 矩陣（Matrices）
- 集合（Sets）
- 命題的真假（Truth Values）

本習題將探討：

```text
數學物件
    ↓
Python 型別
    ↓
CPython 實作
    ↓
AI 中的數值表示
```

---

## 第一部分：Python 如何表示數學物件？

請查閱 Python 文件或自行測試，完成下表。

| 數學概念 | 型別 | |
|----------|------------|------|
| 自然數 | int| 5 |
| 整數 | int | -3, 0, 42 |
| 有理數 | fractions.Fraction | Fraction(3, 4) |
| 實數 | float |3.14159, Decimal('3.14') |
| 複數 | complex | 3 + 4j|
| 向量 | list | [1, 2, 3] 或 np.array([1, 2, 3]) |
| 矩陣 | list |  或 [[1, 2], [3, 4]] 或 np.array([[1, 2], [3, 4]]) |
| 集合 | set（可變）/ frozenset（不可變） | {1, 2, 3} |
| 命題真假 | bool | True, False |

並回答下列問題：
  
1. 哪些是 Python 的內建型別（Built-in Types）？int float complex list set bool
2. 哪些需要額外模組才能較完整地表示？有理數   向量與矩陣 高精度實數 / 十進位實數
3. Python 的型別分類與數學中的分類是否完全一致？ 完全不一致
 
---

## 第二部分：數學概念與 Python 型別

數學上有下列包含關係：

$$
\mathbb N
\subseteq
\mathbb Z
\subseteq
\mathbb Q
\subseteq
\mathbb R
\subseteq
\mathbb C
$$

請回答：

1. Python 是否為上述每一個數系都提供獨立型別？否。Python 沒有獨立的自然數型別與內建的有理數型別。
2. 為什麼自然數與整數都使用 `int` 表示？避免維護負擔與效能開銷。自然數只是整數的子集，其運算規則與整數完全相同，透過 int 配合值域判斷（如 x > 0）即可，無須增加額外型別。
3. Python 為什麼提供 `complex` 型別？支援工程、科學計算與信號處理。複數在科學與工程領域廣泛應用，直接內建 complex 與語法支援（j）可簡化表達式並提升運算效率。
4. Python 為什麼沒有內建的有理數型別？效能與權衡問題。有理數需持續進行最大公因數（GCD）約分，運算速度極慢；大多數日常與科學計算使用硬體加速的 float（雙精度浮點數）即已足夠。
5. 下列三個物件的型別有何不同？三個物件型別完全不同：

1 的型別是 int（整數）。

1.0 的型別是 float（浮點數 / 實數近似值）。

1+0j 的型別是 complex（複數）。

```python
1
1.0
1+0j
```

---

## 第三部分：CPython 如何實作這些型別？

除了知道 Python 使用哪些型別表示數學物件之外，

我們也希望了解：

> Python 物件在 CPython 中是如何實作的？

請查閱資料，完成下表。

| 數學概念 | Python 型別 | CPython 實作 | 主要特點 |
|----------|------------|-------------|----------|
| 自然數 | `int` | PyLongObject | 在 CPython 中無獨立型別，直接採用 int 實作。 |
| 整數 | `int` |PyLongObject | 任意精度整數（Arbitrary Precision）。內部使用陣列（digit ob_digit[]）分段儲存大數，不受 C 語言原生 long 的位元數限制，記憶體隨數值大小動態擴展。 |
| 實數 | `float` |PyLongObject | 固定 64 位元。內部封裝 C 語言的 double 型別，遵循 IEEE 754 雙精度浮點數標準，運算直接依賴硬體 CPU 處理，速度快但有精度誤差 |
| 複數 | `complex` | PyComplexObject | 雙 double 結構。內部直接包含兩個 C double 變數（real 與 imag），分別紀錄實部與虛部。 |
| 集合 | `set` | PySetObject | 雜湊表（Hash Table）。元素必須為可雜湊物件（Hashable），搜尋、新增、刪除的平均時間複雜度為 $O(1)$，不保證元素順序。 |
| 命題真假 | `bool` |PyLongObject |int 的子類別。全域僅有兩個單例物件（Py_True 與 Py_False），實質上就是整數 1 與 0 的特殊封裝。 |



## 第四部分：深入探討各種型別

請針對第三部分中的每一種型別回答：

1. 它是否為 Python 的內建型別？
2. 對應的 CPython 物件結構名稱為何？
3. 它如何儲存資料？
4. 是否有固定大小限制？
5. 若資料規模持續增加，如何擴充儲存空間？
1 (int)
1.是
2.PyLongObject
3.以 base-$2^{30}$ 為單位的 ob_digit 陣列分段儲存
4.無限制（僅受系統可用記憶體限制）
5.動態重新分配記憶體並擴充 ob_digit 陣列長度

1.0 (float)
1.是
2.PyFloatObject
3.封裝 C 語言原生 double 變數（IEEE 754 標準）
4.有限制（固定 64 位元，指數範圍約 $10^{-308} \sim 10^{308}$）
5.無法擴充，超出範圍即報 OverflowError 溢位


1+0j (complex)
1.是
2.PyComplexObject
3.內建 Py_complex 結構，包含兩個 C double（實部與虛部）
4.有限制（實部與虛部各固定 64 位元，共 128 位元）
5.無法擴充，實部或虛部超出範圍即報 OverflowError 溢位
---


## 第五部分：Python 整數與浮點數

執行下列程式：

```python
print(10**100)
print(10**1000)
```

回答：

1. 為什麼 Python 能表示如此大的整數？Python 的 int 採用動態記憶體分配機制。它不會將整數固定限制在硬體的 32 位元或 64 位元空間內，而是依照數字大小，動態建立陣列來分段儲存數字的各個位元。
2. Python 的 `int` 是否具有固定的位元數限制？沒有。Python 3 的 int 沒有固定的 32 位元或 64 位元限制，位元數會隨著數值增加自動擴充。
3. 什麼是 Arbitrary Precision（任意精度）？指計算機系統在處理數值時，不設定固定的位元長度上限。只要記憶體足夠，系統就能精確表示並運算任意長度、任意位數的數字，不會發生傳統數值型別的算術溢位（Overflow）。
4. CPython 使用什麼物件來實作 `int`？使用 C 語言結構體 PyLongObject 實作。
5. Python 整數大小真正的限制是什麼？真正的限制僅有電腦的可用記憶體（RAM）大小，以及系統或程式可處理的最大記憶體定址範圍（此外，Python 對於字串與大整數相互轉換時有預設的位數限制 sys.set_int_max_str_digits，但數值運算本身只受記憶體限制）。

---

再執行：

```python
0.1 + 0.2
```

```python
0.1 + 0.2 == 0.3
```

```python
format(0.1, ".20f")
```

回答：

1. 為什麼結果與數學上的答案不同？因為電腦內部採用二進位儲存資料。十進位的小數（如 0.1 與 0.2）轉為二進位時會變成無窮循環小數。電腦受限於 64 位元記憶體空間，必須對其進行裁切與捨入，因此產生微小的截斷誤差，累積後導致 0.1 + 0.2 變成 0.30000000000000004。
2. 浮點數是否能精確表示所有小數？不能
3. 什麼是浮點數的精度（precision）？精確紀錄有效數字的位數
4. 什麼是浮點數的範圍（range）？最大與最小數值區間
5. 精度與範圍有何不同？精度（Precision）：決定數字有多「準」（能精確保留多少位有效數字）。

範圍（Range）：決定數字有多「大」或多「小」（極限數值邊界在哪裡）。

例比：一個數字可以很大（範圍廣），但末尾的細節不準確（精度有限）。

---

## 第六部分：AI 模型中的數值表示

人工智慧模型需要儲存大量參數（Parameters）與權重（Weights）。

請查閱資料並回答：

1. AI 模型中的權重通常使用什麼型別儲存？主要使用低精度浮點數
2. 什麼是 FP64？佔用 64 位元（1 符號、11 指數、52 尾數）。精度極高但記憶體與計算開銷巨大，AI 領域幾乎不用。
3. 什麼是 FP32？佔用 32 位元（1 符號、8 指數、23 尾數）。傳統深度學習標配，精度高但記憶體與頻寬消耗較大。
4. 什麼是 FP16？佔用 16 位元（1 符號、5 指數、10 尾數）。顯存佔用減半、計算加速，但指數範圍小，易發生下溢（Underflow）或上溢
5. 什麼是 BF16？佔用 16 位元（1 符號、8 指數、7 尾數）。保留與 FP32 相同的指數範圍（動態範圍大），不易溢位，現為大語言模型（LLM）主流訓練與推理格式。
6. 為什麼大型 AI 模型不直接使用 Python 的 `int` 作為權重？缺乏小數表示能力 
 無硬體向量加速 記憶體開銷過大
---

## 第七部分：比較不同數值表示方式

完成下表。

| 表示方式 | 位元數 | 是否有誤差 | 主要用途 |
|----------|--------|------------|----------|
| int | 動態 | 無| 計數、索引、控制邏輯、精確整數運算 |
| FP64 | 64 | 有 | 高精度科學計算、物理模擬、金融工程 |
| FP32 | 32 | 有 | 傳統深度學習、通用科學計算、3D 繪圖 Rendering |
| FP16 | 16 | 有 | 混合精度訓練、行動端 GPU 加速、AI 模型推論 |
| BF16 | 16 | 有 | 大型語言模型（LLM）訓練與主流 AI 推論 |

並回答：

1. 哪種表示方式最精確？整數運算
2. 哪種表示方式最節省記憶體？FP16 與 BF16
3. 哪種表示方式最常見於 AI 訓練？BF16
4. 哪種表示方式最常見於 AI 推論（Inference）？BF16 / FP16

---

## 第八部分：綜合思考

### 思考題一

Python 官方文件指出：

```python
int
```

具有 Unlimited Precision（無限制精度）。

請回答：

1. Unlimited Precision 是什麼意思？不受固定硬體暫存器位元數
2. 它是否代表可以儲存無限大的整數？否
3. CPython 如何實現這項特性？變長陣列儲存

---

### 思考題二

比較：

```python
int
```

與

```python
float
```

回答：

1. 為什麼 `int` 幾乎沒有固定上限？int 採用軟體層面
2. 為什麼 `float` 卻存在範圍限制？float 是為了計算效能而設計的
3. 為什麼 `float` 可能產生誤差？進位制轉換問題

---

### 思考題三

比較下列兩種設計理念：

#### Python 的整數

```text
Arbitrary Precision
正確性優先
```

#### AI 模型常用的 FP16、BF16

```text
Limited Precision
效率優先
```

請說明：

1. 兩種設計理念的差異。Arbitrary Precision（Python int：正確性優先）

核心哲學：「結果必須 100% 準確，不能丟失任何一位數。」

運作機制：犧牲運算速度與記憶體空間，換取絕對的數值精確性。硬體無法直接運算時，交由軟體層面（CPython）動態管理記憶體與分段計算。

Limited Precision（AI FP16/BF16：效率優先）

核心哲學：「在容忍極小容錯的前提下，追求極致的速度與最小的記憶體開銷。」

運作機制：將數據長度嚴格限制在 16 位元，全面交由硬體專用單元（如 GPU Tensor Core）進行大規模矩陣平行計算。
2. 各自的優點與缺點。Arbitrary Precisionint (正確性優先) | 1. 零誤差：無算術溢位（Overflow）或捨入誤差。適用嚴肅計算：加密學、金融結算、組合數學。 | 1. 速度極慢：無法利用 CPU/GPU 硬體加速。記憶體膨脹：大數運算會快速消耗大量記憶體。 || Limited PrecisionFP16 / BF16 (效率優先) | 1. 極速運算：硬體平行處理，吞吐量極高。極省顯存：顯存需求降低 50%～75%。能耗極低：減少數據傳輸帶寬的電力消耗。 | 1. 存在誤差：存在有效位數截斷與溢位風險。不適用精密邏輯：無法用於精確計數或嚴格符號代數。 |
3. 為什麼大型語言模型通常選擇較低精度的表示方式？神經網絡具備強大的「抗噪性（Robustness）」：深度學習本質上是高度冗餘的統計機率模型，權重微小的精度損失（如 $0.0001234$ 變成 $0.000123$）會被網絡層數與矩陣乘法吸收，幾乎不影響最終輸出的語意品質。突破記憶體頻寬牆（Memory Bandwidth Wall）：百億至千億參數的模型（如 70B 參數），若以 FP32 載入需 280GB 顯存；改用 BF16/FP16 僅需 140GB，能直接少用一半的 GPU，大幅提升吞吐量。極致壓榨 GPU 硬體效能：現代 GPU（如 NVIDIA H100/B200）的 Tensor Core 針對 FP16/BF16 進行了硬體級電路優化，其算力（TFLOPS）通常是 FP64 的數十倍以上。
4. 如果所有 AI 模型都改用 FP64，可能會產生哪些問題？顯存爆炸（Memory Explosion）：
每個參數需要 8 位元組（Bytes）。一個 700 億參數（70B）的模型光是儲存權重就需要 560GB 顯存，訓練時的梯度與優化器狀態（Optimizer States）更會暴增至數個 TB，極少數機構才負擔得起。

算力與訓練時間暴增：
GPU 處理 FP64 的電路單元（FP64 Core）數量遠少於 FP16/BF16 Tensor Core。整體運算速度會下降 10 倍至 30 倍以上，導致原本訓練 1 個月的模型變成要訓練數年。

電費與硬體成本無法承受：
數據傳輸與計算的能耗大幅增加，AI 資料中心的建置與營運成本將呈指數級飆升，極大地阻礙 AI 技術的普及與應用。

---

### 挑戰題

請用自己的話解釋：

```python
x = 10**1000
```

為什麼 Python 可以正常運作；

而大型 AI 模型卻通常選擇使用：

```text
FP32
FP16
BF16
```

而不是任意精度整數或高精度浮點數？

請從：

- 記憶體使用量
- 計算速度
- 數值精度
- GPU 運算效率

等角度進行討論。